In [19]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from tqdm import tqdm

In [ ]:
# PREDICTION VARIABLES
N_SEEDS = 10
N_MODELS = 100
TEST_SIZE = 0.20
VALIDATION_SIZE = 0.25

MODEL_CLASS = "nn" #lr,nn,rf
BOOTSTRAP_SIZE = 0.50
QUALIFICATION_COL = "qualified_gagne_2"

fp_data = "../data/obermeyer/obermeyer_data_cleaned.csv"
fp_out = f"../data/predictions/test_bootstrap_{MODEL_CLASS}_{QUALIFICATION_COL}.csv"

In [ ]:
features = ['dem_female', 'dem_age_band_18-24_tm1', 'dem_age_band_25-34_tm1', 'dem_age_band_35-44_tm1', 'dem_age_band_45-54_tm1',
            'dem_age_band_55-64_tm1', 'dem_age_band_65-74_tm1', 'dem_age_band_75+_tm1', 'hypertension_elixhauser_tm1', 'cost_dialysis_tm1',
            'cost_emergency_tm1', 'cost_home_health_tm1', 'cost_ip_medical_tm1', 'cost_ip_surgical_tm1', 'cost_laboratory_tm1',
            'cost_op_primary_care_tm1', 'cost_op_specialists_tm1', 'cost_op_surgery_tm1', 'cost_other_tm1', 'cost_pharmacy_tm1',
            'cost_physical_therapy_tm1', 'cost_radiology_tm1', 'gagne_sum_tm1']
other_variables = ['person_id', 'gagne_sum_t', 'cost_t']

In [ ]:
df = pd.read_csv(fp_data)
X = df[features+other_variables]
y = df[QUALIFICATION_COL]

In [23]:
def get_bootstrap_model(X_train, y_train, random_seed, bootstrap_size=BOOTSTRAP_SIZE):
    np.random.seed(random_seed)
    idx = np.random.permutation(len(X_train))[:int(len(X_train)*bootstrap_size)]
    if MODEL_CLASS == "lr":
        model = LogisticRegression()
    elif MODEL_CLASS == "rf":
        model = RandomForestClassifier(random_state=random_seed)
    elif MODEL_CLASS == "nn":
        model = MLPClassifier(random_state=random_seed, max_iter=1000)
    model.fit(X_train[idx], y_train[idx])
    return model

In [ ]:
output = []
for random_seed in range(N_SEEDS):
    print("random seed", random_seed)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=random_seed)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=VALIDATION_SIZE, random_state=random_seed)

    cost = X_test["cost_t"].to_numpy()
    gagne = X_test["gagne_sum_t"].to_numpy()
    person_id = X_test['person_id'].to_numpy()
    
    X_train = X_train.drop(columns=other_variables).to_numpy().copy()
    y_train = y_train.to_numpy().copy()
    X_val = X_val.drop(columns=other_variables).to_numpy().copy()
    y_val = y_val.to_numpy().copy()
    X_test = X_test.drop(columns=other_variables).to_numpy().copy()
    y_test = y_test.to_numpy().copy()

    predictions = {}
    training_loss = {}
    validation_loss = {}
    for i in tqdm(range(N_MODELS)):
        model = get_bootstrap_model(X_train, y_train, random_seed=i, bootstrap_size=BOOTSTRAP_SIZE)        
        predictions[f'm_{i+1}'] = model.predict_proba(X_test)[:, 1]
        training_loss[f'm_{i+1}'] = log_loss(y_train, model.predict_proba(X_train))
        validation_loss[f'm_{i+1}'] = log_loss(y_val, model.predict_proba(X_val))
                            
    predictions_df = pd.concat([
        pd.DataFrame(predictions),
        pd.DataFrame(training_loss, index=[0]),
        pd.DataFrame(validation_loss, index=[0])]).reset_index(drop=True)
    
    predictions_df["y"] = np.concatenate([y_test, [np.nan, np.nan]])
    predictions_df["person_id"] = np.concatenate([person_id, [-2, -1]]) # -1 indicates validation loss, -2 indicates training loss
    predictions_df['cost_t'] = np.concatenate([cost, [np.nan, np.nan]]) 
    predictions_df['gagne_sum_t'] = np.concatenate([gagne, [np.nan, np.nan]])
    predictions_df["seed"] = random_seed

    output.append(predictions_df)
    
output = pd.concat(output)
for c in output.columns:
    if c.startswith("m_"):
        output[c] = output[c].astype('float32')

random seed 0


100%|██████████| 100/100 [03:48<00:00,  2.28s/it]


random seed 1


100%|██████████| 100/100 [03:51<00:00,  2.31s/it]


random seed 2


...

100%|██████████| 100/100 [03:55<00:00,  2.35s/it]


random seed 8


100%|██████████| 100/100 [04:03<00:00,  2.43s/it]


random seed 9


100%|██████████| 100/100 [03:45<00:00,  2.25s/it]


In [25]:
output

,m_1,m_2,m_3,m_4,m_5,m_6,m_7,m_8,m_9,m_10,...,m_96,m_97,m_98,m_99,m_100,y,person_id,cost_t,gagne_sum_t,seed
0,0.035113,0.040755,0.028409,0.041073,0.046362,0.040074,0.040682,0.028767,0.044935,0.036925,...,0.038566,0.046339,0.047128,0.024305,0.029658,0.0,2545,5300.0,1.0,0
1,0.143790,0.157290,0.196110,0.155157,0.248242,0.227302,0.192770,0.204266,0.225264,0.220997,...,0.227905,0.163499,0.157760,0.197697,0.200410,0.0,8198,2700.0,2.0,0
2,0.011021,0.026455,0.023445,0.024036,0.031154,0.030750,0.030714,0.042554,0.033327,0.026632,...,0.032200,0.031878,0.025153,0.027037,0.033447,0.0,46461,5200.0,0.0,0
3,0.002937,0.001953,0.003318,0.004783,0.003746,0.003927,0.005201,0.007541,0.002997,0.006374,...,0.005857,0.004652,0.003660,0.006997,0.004877,0.0,30620,1300.0,0.0,0
4,0.423905,0.214177,0.252425,0.301115,0.410133,0.215405,0.253378,0.231303,0.194930,0.353174,...,0.222590,0.338706,0.201176,0.221148,0.212825,0.0,47418,2200.0,2.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9754,0.819320,0.864286,0.792453,0.761930,0.789185,0.716410,0.788992,0.810481,0.789561,0.845121,...,0.776585,0.868225,0.717998,0.845391,0.722574,1.0,28651,3500.0,3.0,9
9755,0.005675,0.005891,0.005064,0.003251,0.003820,0.005236,0.006177,0.009450,0.007524,0.008346,...,0.004043,0.005521,0.006607,0.007911,0.005735,0.0,21611,0.0,0.0,9
9756,0.925657,0.938146,0.937828,0.947511,0.920396,0.909361,0.924316,0.905645,0.927668,0.923067,...,0.923434,0.945326,0.897439,0.916194,0.931056,1.0,965,46400.0,8.0,9
9757,0.219457,0.220251,0.219690,0.221811,0.218590,0.219099,0.219791,0.219504,0.218739,0.219297,...,0.219987,0.219874,0.220006,0.220145,0.219014,NaN,-2,NaN,NaN,9


In [ ]:
output.to_csv(fp_out, index=False)